# Đánh giá tác động của các yếu tố lịch và sự kiện trong dự báo nhu cầu bán lẻ trên các mô hình học máy
*Impact of calendar and event factors on retail demand forecasting across machine-learning model families (M5, Walmart).*

**Phiên bản 4.** Thiết kế mới theo góp ý của giảng viên hướng dẫn:

| Thành phần | Lựa chọn |
|---|---|
| Dữ liệu | M5, cửa hàng CA_1, nhóm FOODS, 100 mặt hàng lấy mẫu phân tầng theo doanh số (giống v2) |
| Bài toán | Dự báo nhu cầu **1 ngày tới** cho từng mặt hàng; chia theo thời gian 70 / 15 / 15 |
| Ba nhóm yếu tố lịch | **G1 nội sinh** (thứ, ngày, tuần, tháng, mùa, cuối tuần) · **G2 ngoại sinh** (sự kiện, lễ hội) · **G3 chính sách** (trợ cấp SNAP) |
| Bộ biến | S0 = chỉ lịch sử · S1 = +G1 · S2 = +G2 · S3 = +G3 (lồng nhau) · D1 = S3 − G1 · D2 = S3 − G2 (bỏ một nhóm) |
| Bốn họ mô hình, mỗi họ một đại diện | Thống kê: **ARIMA** có biến ngoại sinh · Cây: **LightGBM** · Hồi quy: **LSTM** · Kiến trúc mới: **N-BEATS** (NBEATSx) |
| Đánh giá | WAPE, RMSE, RMSSE, Bias trên cùng 28.700 dòng kiểm tra; Diebold–Mariano; bootstrap khối tuần; phân tầng theo loại ngày, SNAP |

Giá bán **không** thuộc thiết kế này (không phải yếu tố lịch). Nhãn loại ngày 5 mức chỉ dùng để phân tầng đánh giá, không làm đặc trưng.

**Cách chạy.** Runtime → Change runtime type → **T4 GPU**, rồi Run all. Mỗi mô hình lưu dự báo vào `PRED_DIR`; đặt `USE_DRIVE = True` để lưu sang Google Drive, chạy lại sẽ đọc lại thay vì huấn luyện lại. Thời gian ước tính trên T4: ARIMA 20–60 phút (CPU), LightGBM vài phút, LSTM 1–2 giờ, N-BEATS 30–60 phút.

## 0. Cấu hình

In [ ]:
import os, gc, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# --- DỮ LIỆU M5 ---------------------------------------------------------------
# Notebook tự tìm 3 tệp calendar.csv, sales_train_evaluation.csv, sell_prices.csv theo thứ tự:
#   (1) DATA_DIR bạn đặt dưới đây; (2) /content/M5_Data; (3) Drive: MyDrive/M5_Data; (4) tải bằng kagglehub.
# Nếu kagglehub báo lỗi đăng nhập: thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets (biểu tượng chìa khóa).
DATA_DIR = None
KAGGLE_DATASET = "aryayadav0513/m5-forecasting-accuracy"

def find_m5_dir(roots):
    for root in roots:
        if root and os.path.isdir(root):
            for dirpath, _, files in os.walk(root):
                if "calendar.csv" in files and "sell_prices.csv" in files:
                    return dirpath
    return None

DATA_DIR = find_m5_dir([DATA_DIR, "/content/M5_Data", "/content/drive/MyDrive/M5_Data"])
if DATA_DIR is None:
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    try:
        from google.colab import userdata
        for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                _v = userdata.get(_k)
                if _v:
                    os.environ[_k] = _v
            except Exception:
                pass
    except ImportError:
        pass
    print("Đang tải M5 bằng kagglehub:", KAGGLE_DATASET)
    DATA_DIR = find_m5_dir([kagglehub.dataset_download(KAGGLE_DATASET)])
if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy calendar.csv và sell_prices.csv. Tải 3 tệp M5 vào /content/M5_Data rồi chạy lại ô này.")
SALES_FILE = next((f for f in ["sales_train_evaluation.csv", "sales_train_validation.csv"]
                   if os.path.exists(os.path.join(DATA_DIR, f))), None)
if SALES_FILE is None:
    raise FileNotFoundError(f"Không có sales_train_evaluation.csv trong {DATA_DIR}")
if SALES_FILE != "sales_train_evaluation.csv":
    print("CẢNH BÁO: chỉ có sales_train_validation.csv (đến d_1913), giai đoạn test sẽ khác báo cáo.")
print("Dữ liệu M5:", DATA_DIR, "| tệp doanh số:", SALES_FILE)

OUT_DIR = "/content/outputs_v4" if os.path.isdir("/content") else os.path.abspath("outputs_v4")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

STORE_ID  = "CA_1"
CAT_ID    = "FOODS"
N_ITEMS   = 100            # items sampled (stratified by sales volume); 100-200 runs fine on Colab
N_BINS    = 5              # number of volume strata
SEED      = 42

LOOKBACK  = 28             # LSTM input window (days)
HORIZON   = 1              # main experiment = 1-day-ahead (do NOT change; use the extension section for H>1)
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

# ---------------- thiết kế v4
SETS_NESTED = ["S0", "S1", "S2", "S3"]
RUN = {"ARIMA": True, "LightGBM": True, "LSTM": True, "N-BEATS": True}
DROP_ONE = {"ARIMA": True, "LightGBM": True, "LSTM": False, "N-BEATS": True}   # thêm D1, D2 (LSTM tốn thời gian)
RUN_RAW_DATE_DEMO = True       # minh họa: dùng thẳng ngày tháng (chỉ số ngày, năm) thay vì phân rã

# ARIMA
ARIMA_CANDIDATES = [(1, 0, 0), (1, 0, 1), (2, 0, 1)]    # chọn theo AIC trên tập huấn luyện
ARIMA_N_JOBS = -1

# LSTM (kiến trúc hai nhánh như v2; chọn số đơn vị từ 3 cấu hình trên tập kiểm định với bộ S3)
SEEDS = [42, 7, 2024]
EPOCHS, BATCH_SIZE, PATIENCE = 40, 256, 6
LSTM_GRID = [(32, 16), (64, 32), (128, 64)]
DROPOUT = 0.2
LSTM_LOSS = "poisson"

# N-BEATS (NBEATSx); chọn độ rộng MLP từ 3 cấu hình trên tập kiểm định với bộ S3
NB_SEEDS = [42, 7, 2024]
NB_GRID = [128, 256, 512]
NB_PARAMS = dict(input_size=LOOKBACK, max_steps=1000, val_check_steps=50, early_stop_patience_steps=5)

N_BOOT = 2000
USE_DRIVE = False
# --------------------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED)
STATE = STORE_ID.split("_")[0]
assert HORIZON == 1
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PRED_DIR = "/content/drive/MyDrive/M5_project/preds_v4"
else:
    PRED_DIR = os.path.join(OUT_DIR, "preds_v4")
os.makedirs(PRED_DIR, exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, name), dpi=150, bbox_inches="tight")
    plt.show()

## 1. Hàm dùng chung
Chép nguyên văn từ `src/m5cal/` của repo (đã có kiểm thử trong `tests/`): nhóm yếu tố lịch, ARIMAX, NBEATSx, bảng tác động theo nhóm, kiểm định và bootstrap.

In [ ]:
# ----- src/m5cal/supplementary.py -----
"""Thí nghiệm bổ sung cho thí nghiệm chính (notebook v3: CA_1 FOODS, 100 mặt hàng, dự báo 1 ngày tới).

Mọi hàm ở đây chỉ cần numpy/pandas/scipy và làm việc trên khung `test_eval` của notebook:
một dòng cho mỗi (item_id, date), cột `demand`, `date`, `date_type`, ... và một cột dự báo cho mỗi mô hình.
Notebook v3 chép nguyên văn tệp này vào một ô, nên mã trong notebook và mã được kiểm thử là một.

Nội dung
- wape, bias, dm_test_daily      : giống hệt định nghĩa trong notebook v2 (để số liệu khớp)
- dm_pairs_table                 : bảng Diebold–Mariano cho nhiều cặp mô hình
- block_bootstrap_wape_diff      : khoảng tin cậy cho chênh lệch WAPE, lấy mẫu lại theo khối tuần
- permutation_importance         : hoán vị theo nhóm biến, dùng chung cho LightGBM (DataFrame) và LSTM (mảng)
- permute_frame_columns, permute_array_channels : bộ hoán vị cho hai loại đầu vào
- Timer, add_ensemble            : đo thời gian chạy; trung bình dự báo nhiều mô hình
"""

import time
from typing import Callable, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy import stats


# --------------------------------------------------------------------------- độ đo (giống notebook v2)
def wape(y, p) -> float:
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    return float(np.abs(y - p).sum() / (np.abs(y).sum() + 1e-8))


def bias(y, p) -> float:
    return float(np.mean(np.asarray(p, float) - np.asarray(y, float)))


def dm_test_daily(df: pd.DataFrame, col1: str, col2: str, loss: str = "abs", lag: int = 7,
                  y_col: str = "demand", date_col: str = "date") -> tuple[float, float]:
    """Diebold–Mariano trên chênh lệch tổn thất cộng gộp theo ngày, phương sai Bartlett đến độ trễ `lag`.

    stat > 0 nghĩa là col1 có tổn thất CAO hơn (kém hơn) col2. Giống hàm dm_test của notebook v2.
    """
    e1 = df[y_col] - df[col1]
    e2 = df[y_col] - df[col2]
    if loss == "abs":
        l1, l2 = e1.abs(), e2.abs()
    elif loss == "sq":
        l1, l2 = e1 ** 2, e2 ** 2
    else:
        raise ValueError("loss phải là 'abs' hoặc 'sq'")
    d = (l1 - l2).groupby(df[date_col]).sum().to_numpy()
    n, dm = len(d), d.mean()
    var = np.var(d)
    for k in range(1, lag + 1):
        var += 2 * (1 - k / (lag + 1)) * np.mean((d[k:] - dm) * (d[:-k] - dm))
    if var <= 0:
        return float("nan"), float("nan")
    stat = dm / np.sqrt(var / n)
    return float(stat), float(2 * (1 - stats.norm.cdf(abs(stat))))


def dm_pairs_table(df: pd.DataFrame, pairs: Sequence[tuple[str, str]], losses=("abs", "sq"),
                   lag: int = 7, alpha: float = 0.05) -> pd.DataFrame:
    rows = []
    for a, b in pairs:
        if a not in df.columns or b not in df.columns:
            continue
        for loss in losses:
            s, p = dm_test_daily(df, a, b, loss, lag)
            rows.append({"model 1": a, "model 2": b, "loss": loss, "DM stat": round(s, 3),
                         "p-value": round(p, 4), f"significant ({int(alpha * 100)}%)": bool(p < alpha),
                         "better": b if s > 0 else a})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- bootstrap theo khối tuần
def _week_key(dates: pd.Series) -> np.ndarray:
    d = pd.to_datetime(dates)
    return (d - pd.to_timedelta(d.dt.dayofweek, unit="D")).dt.normalize().to_numpy()


def block_bootstrap_wape_diff(df: pd.DataFrame, col_a: str, col_b: str, group_col: str | None = None,
                              n_boot: int = 2000, seed: int = 0, alpha: float = 0.05,
                              y_col: str = "demand", date_col: str = "date") -> pd.DataFrame:
    """Khoảng tin cậy cho WAPE(col_b) − WAPE(col_a), tính bằng điểm phần trăm.

    Lấy mẫu lại có hoàn lại các TUẦN lịch (khối 7 ngày) để giữ tương quan trong tuần; mọi dòng của tuần
    được chọn đi cùng nhau. Khi có `group_col` (ví dụ `date_type`), WAPE được tính riêng trong từng nhóm
    trên cùng một mẫu tuần. Giá trị âm nghĩa là col_b tốt hơn col_a.

    p_two_sided là tỷ lệ bootstrap hai phía quanh 0 (xấp xỉ, dùng để tham khảo cùng khoảng tin cậy).
    """
    rng = np.random.default_rng(seed)
    work = pd.DataFrame({
        "week": _week_key(df[date_col]),
        "g": df[group_col].to_numpy() if group_col else "Tất cả",
        "ae_a": (df[y_col] - df[col_a]).abs().to_numpy(float),
        "ae_b": (df[y_col] - df[col_b]).abs().to_numpy(float),
        "y": df[y_col].abs().to_numpy(float),
    })
    agg = work.groupby(["week", "g"], sort=True)[["ae_a", "ae_b", "y"]].sum()
    weeks = agg.index.get_level_values(0).unique()
    groups = list(pd.unique(work["g"]))
    # ma trận tuần × nhóm cho ba tổng
    cube = {k: agg[k].unstack("g").reindex(index=weeks, columns=groups).fillna(0.0).to_numpy() for k in ("ae_a", "ae_b", "y")}
    W = len(weeks)
    idx = rng.integers(0, W, size=(n_boot, W))
    out = []
    for j, g in enumerate(groups):
        A, B, Y = cube["ae_a"][:, j], cube["ae_b"][:, j], cube["y"][:, j]
        if Y.sum() <= 0:
            continue
        point = 100 * (B.sum() / Y.sum() - A.sum() / Y.sum())
        ys = Y[idx].sum(axis=1)
        ok = ys > 0
        diffs = 100 * (B[idx].sum(axis=1)[ok] - A[idx].sum(axis=1)[ok]) / ys[ok]
        lo, hi = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
        p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
        out.append({"group": g, "model a": col_a, "model b": col_b,
                    "WAPE a (%)": round(100 * A.sum() / Y.sum(), 2), "WAPE b (%)": round(100 * B.sum() / Y.sum(), 2),
                    "diff b-a (pp)": round(point, 2), "CI low": round(float(lo), 2), "CI high": round(float(hi), 2),
                    "p_two_sided": round(float(min(p, 1.0)), 4),
                    "CI excludes 0": bool(lo > 0 or hi < 0),
                    "weeks with data": int((Y > 0).sum()),
                    "days": int(df.loc[(df[group_col] == g) if group_col else slice(None), date_col].nunique())})
    return pd.DataFrame(out)


# --------------------------------------------------------------------------- permutation importance theo nhóm
Permuter = Callable[[object, np.random.Generator], object]


def permute_frame_columns(cols: Sequence[str]) -> Permuter:
    """Hoán vị CÙNG MỘT thứ tự dòng cho mọi cột trong nhóm (giữ quan hệ giữa các cột trong nhóm)."""
    cols = list(cols)

    def _perm(X: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
        Xp = X.copy()
        order = rng.permutation(len(X))
        for c in cols:
            vals = X[c].to_numpy()
            if isinstance(X[c].dtype, pd.CategoricalDtype):
                Xp[c] = pd.Categorical.from_codes(X[c].cat.codes.to_numpy()[order], dtype=X[c].dtype)
            else:
                Xp[c] = vals[order]
        return Xp

    return _perm


def permute_array_channels(spec: Mapping[str, Sequence[int] | None]) -> Permuter:
    """Hoán vị mẫu (trục 0) cho các kênh được chọn của nhiều mảng đầu vào, dùng chung một thứ tự.

    spec: {tên đầu vào: danh sách chỉ số kênh ở trục cuối, hoặc None = toàn bộ mảng}.
    Ví dụ nhóm lịch của LSTM-C: {"hist": [1, ..., 15], "future": [0, ..., 14]}.
    """
    spec = {k: (None if v is None else list(v)) for k, v in spec.items()}

    def _perm(inputs: Mapping[str, np.ndarray], rng: np.random.Generator) -> dict:
        n = len(next(iter(inputs.values())))
        order = rng.permutation(n)
        out = dict(inputs)
        for name, ch in spec.items():
            arr = np.array(inputs[name], copy=True)
            if ch is None:
                arr = arr[order]
            else:
                arr[..., ch] = arr[order][..., ch]
            out[name] = arr
        return out

    return _perm


def permutation_importance(predict_fn: Callable[[object], np.ndarray], inputs, y,
                           permuters: Mapping[str, Permuter], metric: Callable = wape,
                           n_repeats: int = 3, seed: int = 0) -> pd.DataFrame:
    """Mức tăng của `metric` (điểm phần trăm nếu metric là wape) khi hoán vị từng nhóm biến.

    predict_fn nhận đúng kiểu `inputs` (DataFrame cho LightGBM, dict mảng cho LSTM) và trả về dự báo.
    Dự báo được cắt tại 0 như trong notebook.
    """
    rng = np.random.default_rng(seed)
    y = np.asarray(y, float)
    base = metric(y, np.clip(np.asarray(predict_fn(inputs), float).ravel(), 0, None))
    rows = []
    for name, perm in permuters.items():
        scores = []
        for _ in range(n_repeats):
            pred = np.clip(np.asarray(predict_fn(perm(inputs, rng)), float).ravel(), 0, None)
            scores.append(metric(y, pred))
        scores = np.asarray(scores)
        rows.append({"group": name, "base": 100 * base, "permuted mean": 100 * scores.mean(),
                     "increase (pp)": 100 * (scores.mean() - base), "increase std (pp)": 100 * scores.std(ddof=0),
                     "repeats": n_repeats})
    return pd.DataFrame(rows).sort_values("increase (pp)", ascending=False).round(3).reset_index(drop=True)


# --------------------------------------------------------------------------- tiện ích
class Timer:
    """with Timer() as t: ...;  t.seconds"""

    def __enter__(self):
        self._t0 = time.perf_counter()
        self.seconds = float("nan")
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self._t0
        return False


def add_ensemble(df: pd.DataFrame, cols: Sequence[str], name: str) -> pd.DataFrame:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Thiếu cột dự báo: {missing}")
    df[name] = df[list(cols)].mean(axis=1)
    return df


# ----- src/m5cal/calendar_groups.py -----
"""Ba nhóm yếu tố lịch dùng trong thiết kế mới (phiên bản 0.3.0).

Ngày tháng là một biến liên tục, tăng mãi theo thời gian: đưa thẳng nó vào mô hình thì mô hình chỉ thấy
"thời điểm thứ bao nhiêu", không thấy các chu kỳ lặp lại hay ngữ cảnh của ngày. Vì vậy thông tin ngày tháng
được phân rã và gắn ngữ cảnh thành ba nhóm thuộc tính:

- G1 · Nội sinh (phái sinh từ ngày): thứ, ngày trong tháng, tuần trong năm, tháng, mùa, cuối tuần.
- G2 · Ngoại sinh (sự kiện, lễ hội): loại sự kiện, ngày lễ, trước sự kiện, khoảng cách tới / từ sự kiện.
- G3 · Chính sách trợ cấp (SNAP): ngày phát SNAP và vị trí của ngày trong đợt phát.

Mỗi nhóm có ba cách biểu diễn, tùy mô hình:
- "tree"   : giá trị số nguyên / nhị phân cho LightGBM.
- "linear" : biến giả (one-hot) cho ARIMA có biến ngoại sinh (mô hình tuyến tính).
- "neural" : mã hóa sin–cos và biến đã chuẩn hóa cho LSTM và N-BEATS.

Bộ đặc trưng (cấu hình ablation):
- S0 = chỉ lịch sử nhu cầu
- S1 = S0 + G1, S2 = S1 + G2, S3 = S2 + G3 (lồng nhau, đo tác động tăng thêm)
- D1 = S3 − G1, D2 = S3 − G2 (bỏ một nhóm khỏi bộ đầy đủ, đo tác động biên; S3 − G3 chính là S2)
"""

import numpy as np
import pandas as pd

GROUP_NAMES = {"G1": "Nội sinh (thứ, ngày, tháng, mùa)",
               "G2": "Ngoại sinh (sự kiện, lễ hội)",
               "G3": "Chính sách trợ cấp (SNAP)"}
EVENT_TYPES = ["National", "Religious", "Cultural", "Sporting"]
HOLIDAY_TYPES = ["National", "Religious"]
DATE_TYPES = ["Weekday", "Weekend", "Pre_Event", "Other_Event", "Holiday"]
SEASONS = ["Winter", "Spring", "Summer", "Autumn"]          # bắc bán cầu: 12-2, 3-5, 6-8, 9-11

SET_GROUPS = {"S0": [], "S1": ["G1"], "S2": ["G1", "G2"], "S3": ["G1", "G2", "G3"],
              "D1": ["G2", "G3"], "D2": ["G1", "G3"]}
NESTED = ["S0", "S1", "S2", "S3"]


def _cyc(x, period):
    a = 2 * np.pi * np.asarray(x, float) / period
    return np.sin(a), np.cos(a)


def build_calendar_groups(calendar: pd.DataFrame, state: str = "CA", event_window: int = 14,
                          pre_event_days: tuple[int, int] = (1, 2)) -> pd.DataFrame:
    """Từ calendar.csv của M5, tạo mọi thuộc tính của ba nhóm (mỗi dòng một ngày).

    Chỉ dùng thông tin có trong lịch của chính ngày đó và các ngày sự kiện đã công bố, nên mọi thuộc tính
    đều biết trước tại thời điểm dự báo.
    """
    c = calendar.copy()
    c["date"] = pd.to_datetime(c["date"]).astype("datetime64[ns]")
    c = c.sort_values("date").reset_index(drop=True)

    # ---------------- G1: phân rã ngày tháng
    c["day_of_week"] = c["date"].dt.dayofweek                       # 0 = thứ Hai
    c["day_of_month"] = c["date"].dt.day
    c["week_of_year"] = c["date"].dt.isocalendar().week.astype(int)
    c["month"] = c["date"].dt.month
    c["season"] = ((c["month"] % 12) // 3).astype(int)              # 0 Đông, 1 Xuân, 2 Hạ, 3 Thu
    c["is_weekend"] = (c["day_of_week"] >= 5).astype(int)
    c["dow_sin"], c["dow_cos"] = _cyc(c["day_of_week"], 7)
    c["dom_sin"], c["dom_cos"] = _cyc(c["day_of_month"] - 1, 31)
    c["woy_sin"], c["woy_cos"] = _cyc(c["week_of_year"] - 1, 53)
    c["month_sin"], c["month_cos"] = _cyc(c["month"] - 1, 12)
    for k in range(1, 7):
        c[f"dow_{k}"] = (c["day_of_week"] == k).astype(int)          # thứ Hai làm mốc
    for k in range(2, 13):
        c[f"month_{k}"] = (c["month"] == k).astype(int)             # tháng 1 làm mốc
    for k, s in enumerate(SEASONS):
        c[f"season_{s}"] = (c["season"] == k).astype(int)

    # ---------------- G2: sự kiện, lễ hội
    t1 = c.get("event_type_1", pd.Series(index=c.index, dtype=object))
    t2 = c.get("event_type_2", pd.Series(index=c.index, dtype=object))
    for et in EVENT_TYPES:
        c[f"ev_{et}"] = ((t1 == et) | (t2 == et)).astype(int)
    c["has_event"] = (t1.notna() | t2.notna()).astype(int)
    c["is_holiday"] = (t1.isin(HOLIDAY_TYPES) | t2.isin(HOLIDAY_TYPES)).astype(int)
    ev_date = c["date"].where(c["has_event"] == 1)
    c["days_to_next_event"] = (ev_date.bfill() - c["date"]).dt.days.clip(upper=event_window).fillna(event_window)
    c["days_since_last_event"] = (c["date"] - ev_date.ffill()).dt.days.clip(upper=event_window).fillna(event_window)
    lo, hi = pre_event_days
    c["is_pre_event"] = c["days_to_next_event"].between(lo, hi).astype(int)
    c["dts"] = c["days_to_next_event"] / event_window
    c["dsl"] = c["days_since_last_event"] / event_window

    # ---------------- G3: chính sách trợ cấp SNAP của bang
    col = f"snap_{state}"
    c["snap"] = c[col].astype(int) if col in c else 0
    run = c["snap"].groupby((c["snap"] != c["snap"].shift()).cumsum()).cumsum()
    c["snap_run_day"] = np.where(c["snap"] == 1, run, 0).astype(int)  # ngày thứ mấy của đợt phát SNAP
    c["snap_run_scaled"] = c["snap_run_day"] / max(int(c["snap_run_day"].max()), 1)

    # ---------------- nhãn loại ngày: CHỈ dùng để phân tầng đánh giá, không làm đặc trưng
    c["date_type"] = np.select(
        [c["is_holiday"] == 1, c["has_event"] == 1, c["is_pre_event"] == 1, c["is_weekend"] == 1],
        ["Holiday", "Other_Event", "Pre_Event", "Weekend"], default="Weekday")
    return c


# Danh sách cột của từng nhóm, theo cách biểu diễn
GROUP_COLUMNS = {
    "tree": {
        "G1": ["day_of_week", "day_of_month", "week_of_year", "month", "season", "is_weekend"],
        "G2": [f"ev_{e}" for e in EVENT_TYPES] + ["has_event", "is_holiday", "is_pre_event",
                                                  "days_to_next_event", "days_since_last_event"],
        "G3": ["snap", "snap_run_day"],
    },
    "linear": {
        "G1": [f"dow_{k}" for k in range(1, 7)] + [f"month_{k}" for k in range(2, 13)],
        "G2": [f"ev_{e}" for e in EVENT_TYPES] + ["is_pre_event"],
        "G3": ["snap", "snap_run_scaled"],
    },
    "neural": {
        "G1": ["dow_sin", "dow_cos", "dom_sin", "dom_cos", "woy_sin", "woy_cos", "month_sin", "month_cos",
               "is_weekend"] + [f"season_{s}" for s in SEASONS],
        "G2": [f"ev_{e}" for e in EVENT_TYPES] + ["has_event", "is_holiday", "is_pre_event", "dts", "dsl"],
        "G3": ["snap", "snap_run_scaled"],
    },
}


def set_columns(set_name: str, kind: str) -> list[str]:
    """Danh sách cột lịch của một bộ đặc trưng (không gồm biến lịch sử nhu cầu)."""
    if set_name not in SET_GROUPS:
        raise KeyError(f"Bộ không tồn tại: {set_name}; có {list(SET_GROUPS)}")
    cols: list[str] = []
    for g in SET_GROUPS[set_name]:
        cols += GROUP_COLUMNS[kind][g]
    return cols


def all_calendar_columns() -> list[str]:
    cols = []
    for kind in GROUP_COLUMNS.values():
        for g in kind.values():
            cols += g
    return list(dict.fromkeys(cols)) + ["date_type", "season", "snap_run_day"]


# ----- src/m5cal/group_effects.py -----
"""Tác động của từng nhóm yếu tố lịch lên sai số của từng mô hình.

Cột dự báo trong khung đánh giá đặt tên "<mô hình>-<bộ>", ví dụ "LightGBM-S2".
Hai loại tác động:
- tăng thêm (incremental): thêm nhóm theo thứ tự G1 → G2 → G3 (S0→S1, S1→S2, S2→S3);
- biên (marginal): bỏ đúng một nhóm khỏi bộ đầy đủ S3 (D1→S3 cho G1, D2→S3 cho G2, S2→S3 cho G3).
Tác động được tính trên mọi ngày và trên "ngày liên quan" của nhóm (G2: ngày lễ, sự kiện và trước sự kiện;
G3: ngày SNAP; G1: mọi ngày).
"""

import numpy as np
import pandas as pd


COMPARISONS = [("G1", "tăng thêm", "S0", "S1"), ("G2", "tăng thêm", "S1", "S2"), ("G3", "tăng thêm", "S2", "S3"),
               ("G1", "biên", "D1", "S3"), ("G2", "biên", "D2", "S3"), ("G3", "biên", "S2", "S3")]


def relevant_mask(df: pd.DataFrame, group: str) -> pd.Series:
    if group == "G2":
        return df["date_type"].isin(["Holiday", "Other_Event", "Pre_Event"])
    if group == "G3":
        return df["snap"] == 1
    return pd.Series(True, index=df.index)


def group_effects(df: pd.DataFrame, models, y_col: str = "demand", lag: int = 7) -> pd.DataFrame:
    rows = []
    for m in models:
        for g, kind, before, after in COMPARISONS:
            cb, ca = f"{m}-{before}", f"{m}-{after}"
            if cb not in df.columns or ca not in df.columns:
                continue
            for scope, mask in (("mọi ngày", pd.Series(True, index=df.index)), ("ngày liên quan", relevant_mask(df, g))):
                if scope == "ngày liên quan" and g == "G1":
                    continue
                sub = df[mask]
                wb, wa = 100 * wape(sub[y_col], sub[cb]), 100 * wape(sub[y_col], sub[ca])
                s_abs, p_abs = dm_test_daily(sub, cb, ca, "abs", lag)
                s_sq, p_sq = dm_test_daily(sub, cb, ca, "sq", lag)
                rows.append({"model": m, "group": g, "type": kind, "scope": scope, "before": before, "after": after,
                             "days": int(sub["date"].nunique()), "WAPE before (%)": round(wb, 2),
                             "WAPE after (%)": round(wa, 2), "delta (pp)": round(wa - wb, 2),
                             "delta (%)": round(100 * (wa / wb - 1), 2) if wb > 0 else np.nan,
                             "DM abs": round(s_abs, 3), "p abs": round(p_abs, 4),
                             "DM sq": round(s_sq, 3), "p sq": round(p_sq, 4),
                             "helps (5%)": bool(s_abs > 0 and p_abs < 0.05)})
    return pd.DataFrame(rows)


def effect_matrix(effects: pd.DataFrame, kind: str = "biên", scope: str = "mọi ngày",
                  value: str = "delta (%)") -> pd.DataFrame:
    """Bảng mô hình × nhóm (giá trị âm = nhóm làm giảm sai số)."""
    sub = effects[(effects["type"] == kind) & (effects["scope"] == scope)]
    return sub.pivot(index="model", columns="group", values=value)


# ----- src/m5cal/arima_exog.py -----
"""ARIMA có biến ngoại sinh (ARIMAX, dùng SARIMAX của statsmodels) cho dự báo 1 ngày tới, từng mặt hàng.

Cách làm:
1. Chọn bậc (p, q) cho từng mặt hàng bằng AIC trên tập huấn luyện, với mô hình chỉ có lịch sử (S0).
   Bậc này được giữ nguyên cho mọi bộ biến lịch, để chênh lệch giữa các bộ chỉ đến từ biến lịch.
2. Ước lượng tham số trên tập huấn luyện.
3. Giữ nguyên tham số, đưa toàn bộ chuỗi (huấn luyện + kiểm định + kiểm tra) vào bộ lọc Kalman
   (`results.apply`) và lấy dự báo một bước (`get_prediction(start=n_train)`): dự báo cho ngày t chỉ
   dùng doanh số đến ngày t−1 và biến lịch của chính ngày t (biết trước), không huấn luyện lại.

Mô hình được truyền vào qua `model_cls` để kiểm thử được mà không cần statsmodels.
"""

from typing import Iterable, Sequence

import numpy as np
import pandas as pd


def _sarimax():
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
    except ImportError as e:  # pragma: no cover
        raise ImportError("Cần statsmodels (Colab đã cài sẵn): pip install statsmodels") from e
    return SARIMAX


def _fit(model_cls, y, X, order):
    mod = model_cls(y, exog=X, order=order, trend="c",
                    enforce_stationarity=False, enforce_invertibility=False)
    return mod.fit(disp=False, maxiter=200)


def select_order(y_train: np.ndarray, candidates: Sequence[tuple[int, int, int]] = ((1, 0, 0), (1, 0, 1), (2, 0, 1)),
                 model_cls=None) -> tuple[int, int, int]:
    """Bậc có AIC nhỏ nhất trên tập huấn luyện (mô hình không có biến ngoại sinh)."""
    model_cls = model_cls or _sarimax()
    best, best_aic = candidates[0], np.inf
    for order in candidates:
        try:
            aic = _fit(model_cls, np.asarray(y_train, float), None, order).aic
        except Exception:  # bậc không hội tụ thì bỏ qua
            continue
        if np.isfinite(aic) and aic < best_aic:
            best, best_aic = order, aic
    return best


def one_step_predictions(y: np.ndarray, X: np.ndarray | None, n_train: int, order: tuple[int, int, int],
                         model_cls=None) -> np.ndarray:
    """Dự báo một bước cho các vị trí n_train, ..., len(y)−1 với tham số ước lượng trên y[:n_train]."""
    model_cls = model_cls or _sarimax()
    y = np.asarray(y, float)
    if X is not None:
        X = np.asarray(X, float)
        keep = X[:n_train].std(axis=0) > 0                     # bỏ cột hằng trong tập huấn luyện
        X = X[:, keep] if keep.any() else None
    res = _fit(model_cls, y[:n_train], None if X is None else X[:n_train], order)
    res_all = res.apply(y, exog=X)                             # cùng tham số, toàn bộ chuỗi
    pred = np.asarray(res_all.get_prediction(start=n_train).predicted_mean, float)
    if len(pred) != len(y) - n_train:
        raise RuntimeError("Số dự báo không khớp số ngày sau tập huấn luyện")
    return pred


def _arima_item(item, g, exog_cols, train_end, fallback_end, min_train, order, candidates, model_cls,
                item_col, date_col, y_col):
    """Một mặt hàng: trả về (item, bậc, DataFrame dự báo) hoặc (item, None, None) nếu quá ít dữ liệu."""
    g = g.sort_values(date_col)
    y = g[y_col].to_numpy(float)
    n_train = int((g[date_col] <= train_end).sum())
    if n_train < min_train and fallback_end is not None:
        n_train = int((g[date_col] <= fallback_end).sum())
    if n_train < min_train or n_train >= len(g):
        return item, None, None
    order = order or select_order(y[:n_train], candidates, model_cls)
    X = g[exog_cols].to_numpy(float) if exog_cols else None
    try:
        pred = one_step_predictions(y, X, n_train, order, model_cls)
    except Exception:                                            # dự phòng: bỏ biến ngoại sinh
        pred = one_step_predictions(y, None, n_train, order, model_cls)
    out = pd.DataFrame({item_col: item, date_col: g[date_col].to_numpy()[n_train:], "pred": np.clip(pred, 0, None)})
    return item, order, out


def run_arima_panel(df: pd.DataFrame, exog_cols: Iterable[str], train_end, orders: dict | None = None,
                    candidates=((1, 0, 0), (1, 0, 1), (2, 0, 1)), item_col: str = "item_id",
                    date_col: str = "date", y_col: str = "demand", model_cls=None, n_jobs: int = 1,
                    fallback_end=None, min_train: int = 30) -> pd.DataFrame:
    """Chạy ARIMAX cho mọi mặt hàng. Trả về (item_id, date, pred) cho mọi ngày sau mốc ước lượng.

    Mốc ước lượng là `train_end`; mặt hàng lên kệ muộn (ít hơn `min_train` ngày trước `train_end`) được
    ước lượng đến `fallback_end` (cuối tập kiểm định) nếu có, để vẫn có dự báo cho tập kiểm tra.
    `orders` (mặt hàng → bậc) được dùng lại nếu có; nếu không, bậc được chọn bằng AIC và trả về trong
    thuộc tính `.attrs["orders"]` của kết quả để dùng cho các bộ biến khác.
    """
    exog_cols = list(exog_cols)
    train_end = pd.Timestamp(train_end)
    fallback_end = None if fallback_end is None else pd.Timestamp(fallback_end)
    groups = [(item, g[[date_col, y_col] + exog_cols]) for item, g in df.groupby(item_col, sort=False)]
    args = [(item, g, exog_cols, train_end, fallback_end, min_train, (orders or {}).get(item), candidates,
             model_cls, item_col, date_col, y_col) for item, g in groups]
    if n_jobs == 1:
        results = [_arima_item(*a) for a in args]
    else:  # pragma: no cover - song song trên Colab
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=n_jobs)(delayed(_arima_item)(*a) for a in args)
    frames = [r[2] for r in results if r[2] is not None]
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[item_col, date_col, "pred"])
    out.attrs["orders"] = {r[0]: r[1] for r in results if r[1] is not None}
    return out


# ----- src/m5cal/nbeats_exog.py -----
"""N-BEATS có biến ngoại sinh (NBEATSx, thư viện neuralforecast) cho dự báo 1 ngày tới.

NBEATSx (Olivares và cộng sự, 2023) mở rộng N-BEATS (Oreshkin và cộng sự, 2020) để nhận biến biết trước
(`futr_exog_list`). Dự báo cho tập kiểm tra được tạo bằng `cross_validation` với h = 1, bước 1 ngày,
không huấn luyện lại (`refit=False`): mô hình học một lần trên phần trước tập kiểm tra, dùng `val_size`
ngày cuối của phần đó để dừng sớm, rồi dự báo từng ngày của tập kiểm tra.

`nf_cls`, `model_cls`, `loss` được truyền vào để kiểm thử được mà không cần neuralforecast.
"""

import pandas as pd

DEFAULT_PARAMS = dict(input_size=28, mlp_width=256, max_steps=1000, val_check_steps=50,
                      early_stop_patience_steps=5, batch_size=32, windows_batch_size=1024,
                      learning_rate=1e-3, scaler_type="standard", accelerator="auto")


def build_nf_frame(df: pd.DataFrame, exog_cols, id_col: str = "item_id", date_col: str = "date",
                   y_col: str = "demand") -> pd.DataFrame:
    exog_cols = list(exog_cols)
    out = df[[id_col, date_col, y_col] + exog_cols].rename(columns={id_col: "unique_id", date_col: "ds", y_col: "y"})
    out["ds"] = pd.to_datetime(out["ds"])
    out[exog_cols] = out[exog_cols].astype("float32")
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def nbeatsx_cross_validation(df_nf: pd.DataFrame, exog_cols, test_size: int, val_size: int, seed: int = 1,
                             params: dict | None = None, nf_cls=None, model_cls=None, loss=None) -> pd.DataFrame:
    """Trả về (unique_id, ds, pred) cho `test_size` ngày cuối của mỗi chuỗi."""
    p = dict(DEFAULT_PARAMS, **(params or {}))
    if nf_cls is None or model_cls is None or loss is None:  # pragma: no cover - cần neuralforecast
        from neuralforecast import NeuralForecast
        from neuralforecast.losses.pytorch import MSE
        from neuralforecast.models import NBEATSx
        nf_cls, model_cls, loss = nf_cls or NeuralForecast, model_cls or NBEATSx, loss or MSE()
    exog_cols = list(exog_cols)
    # Chuỗi quá ngắn (mặt hàng lên kệ muộn) làm cross_validation báo lỗi: bỏ ra, dòng kiểm tra của chúng
    # được điền bằng mô hình cơ sở khi ghép (xem `attach`).
    lengths = df_nf.groupby("unique_id")["ds"].transform("size")
    short = lengths < test_size + val_size + 1
    if short.any():
        print(f"  NBEATSx: bỏ {df_nf.loc[short, 'unique_id'].nunique()} chuỗi ngắn hơn {test_size + val_size + 1} ngày")
        df_nf = df_nf[~short]
    model = model_cls(h=1, input_size=p["input_size"], futr_exog_list=exog_cols or None,
                      stack_types=["identity", "identity", "identity"], n_blocks=[1, 1, 1],
                      mlp_units=3 * [[p["mlp_width"], p["mlp_width"]]], loss=loss,
                      max_steps=p["max_steps"], val_check_steps=p["val_check_steps"],
                      early_stop_patience_steps=p["early_stop_patience_steps"], batch_size=p["batch_size"],
                      windows_batch_size=p["windows_batch_size"], learning_rate=p["learning_rate"],
                      scaler_type=p["scaler_type"], random_seed=seed, start_padding_enabled=True,
                      enable_progress_bar=False, accelerator=p["accelerator"])
    nf = nf_cls(models=[model], freq="D")
    cv = nf.cross_validation(df=df_nf, n_windows=None, test_size=test_size, val_size=val_size,
                             step_size=1, refit=False)
    cv = cv.reset_index() if "unique_id" not in cv.columns else cv
    name = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y", "index")][0]
    return cv[["unique_id", "ds", name]].rename(columns={name: "pred"})


def attach(test_eval: pd.DataFrame, preds: pd.DataFrame, col: str, id_col: str = "item_id",
           date_col: str = "date", fallback_col: str | None = None, max_missing: float = 0.05) -> pd.DataFrame:
    """Ghép dự báo vào khung đánh giá theo (mặt hàng, ngày).

    Dòng thiếu dự báo (ví dụ mặt hàng lên kệ quá muộn để ước lượng mô hình) được điền bằng cột
    `fallback_col` (mô hình cơ sở) nếu tỉ lệ thiếu không vượt `max_missing`; ngược lại báo lỗi.
    Số dòng đã điền được lưu trong `out.attrs["filled"]`.
    """
    p = preds.rename(columns={"unique_id": id_col, "ds": date_col, "pred": col})
    p[date_col] = pd.to_datetime(p[date_col]).astype("datetime64[ns]")
    p = p.drop_duplicates([id_col, date_col])
    out = test_eval.drop(columns=[col], errors="ignore").merge(p[[id_col, date_col, col]], on=[id_col, date_col], how="left")
    missing = int(out[col].isna().sum())
    if missing:
        frac = missing / len(out)
        if fallback_col is None or fallback_col not in out.columns or frac > max_missing:
            raise ValueError(f"{col}: thiếu dự báo cho {missing} dòng kiểm tra ({100 * frac:.1f}%)")
        print(f"  {col}: {missing} dòng ({100 * frac:.2f}%) không có dự báo, điền bằng {fallback_col}")
        out[col] = out[col].fillna(out[fallback_col])
    out[col] = out[col].clip(lower=0)
    out.attrs = dict(getattr(test_eval, "attrs", {}) or {})
    out.attrs.setdefault("filled", {})[col] = missing
    return out



def cached(col, fn):
    """Đọc dự báo đã lưu cho cột `col` nếu có; nếu không, chạy fn() -> DataFrame(item_id, date, pred) và lưu."""
    path = os.path.join(PRED_DIR, f"{col}.parquet")
    if os.path.exists(path):
        print(f"  {col}: đọc lại từ {path}")
        return pd.read_parquet(path)
    t0 = time.time()
    res = fn()
    attrs = dict(getattr(res, "attrs", {}) or {})
    out = res[["item_id", "date", "pred"]].copy()
    out.attrs = {}
    out.to_parquet(path, index=False)
    out.attrs = attrs
    TIMES.append({"column": col, "sec": round(time.time() - t0, 1)})
    print(f"  {col}: xong sau {time.time() - t0:.0f} giây")
    return out

TIMES = []

## 2. Lịch và ba nhóm yếu tố lịch
Ngày tháng là biến liên tục, tăng mãi theo thời gian; dùng thẳng thì mô hình không thấy chu kỳ lặp lại hay ngữ cảnh của ngày. Vì vậy ngày được phân rã (G1) và gắn ngữ cảnh: sự kiện, lễ hội (G2) và chính sách trợ cấp SNAP (G3). Mỗi nhóm có ba cách biểu diễn: số nguyên cho LightGBM, biến giả cho ARIMA, sin–cos và biến chuẩn hóa cho LSTM và N-BEATS.

In [ ]:
calendar_raw = pd.read_csv(os.path.join(DATA_DIR, "calendar.csv"))
cal = build_calendar_groups(calendar_raw, state=STATE)
CAL_COLS = [c for c in all_calendar_columns() if c in cal.columns]
CAL_KEEP = ["d", "date", "wm_yr_wk"] + list(dict.fromkeys(CAL_COLS))
print("Calendar:", cal.shape, "|", cal["date"].min().date(), "->", cal["date"].max().date())
for kind in ["tree", "linear", "neural"]:
    print(kind, {g: len(c) for g, c in GROUP_COLUMNS[kind].items()})
print(cal["date_type"].value_counts())

## 3. Doanh số, lấy mẫu phân tầng 100 mặt hàng (giống v2)

In [ ]:
sales_path = os.path.join(DATA_DIR, SALES_FILE)
parts = []
for chunk in pd.read_csv(sales_path, chunksize=2000):                   # chunked read -> low RAM
    chunk = chunk[(chunk["store_id"] == STORE_ID) & (chunk["cat_id"] == CAT_ID)]
    if len(chunk):
        parts.append(chunk)
sales_wide = pd.concat(parts, ignore_index=True)
del parts; gc.collect()

day_cols = [c for c in sales_wide.columns if c.startswith("d_")]
sales_wide["total_volume"] = sales_wide[day_cols].sum(axis=1)
pool_all = len(sales_wide)
sales_wide = sales_wide[sales_wide["total_volume"] > 0].reset_index(drop=True)   # drop items that never sell
sales_wide["vol_bin"] = pd.qcut(sales_wide["total_volume"].rank(method="first"), N_BINS, labels=False)

per_bin = N_ITEMS // N_BINS
selected = pd.concat([
    sales_wide[sales_wide["vol_bin"] == b].sample(min(per_bin, (sales_wide["vol_bin"] == b).sum()), random_state=SEED)
    for b in range(N_BINS)
], ignore_index=True)

print(f"{STORE_ID}/{CAT_ID}: {pool_all} items in pool, {len(sales_wide)} with sales -> {len(selected)} selected")
print(selected.groupby("vol_bin")["total_volume"].agg(["count", "min", "median", "max"]))
print("Departments:", selected["dept_id"].value_counts().to_dict())
del sales_wide; gc.collect()

## 4. Chuyển sang dạng dọc, ghép lịch và giá (giá chỉ dùng để loại ngày mặt hàng chưa lên kệ)

In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
sales_long = selected.melt(id_vars=id_cols + ["vol_bin", "total_volume"],
                           value_vars=day_cols, var_name="d", value_name="demand")
sales_long["demand"] = sales_long["demand"].astype("float32")
sales_long = sales_long.merge(cal[CAL_KEEP], on="d", how="left")

item_set = set(selected["item_id"])
price_parts = []
for chunk in pd.read_csv(f"{DATA_DIR}/sell_prices.csv", chunksize=200_000):
    chunk = chunk[(chunk["store_id"] == STORE_ID) & (chunk["item_id"].isin(item_set))]
    if len(chunk):
        price_parts.append(chunk)
prices = pd.concat(price_parts, ignore_index=True)
del price_parts; gc.collect()

sales_long = sales_long.merge(prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
                              on=["store_id", "item_id", "wm_yr_wk"], how="left")
n_before = len(sales_long)
sales_long = sales_long[sales_long["sell_price"].notna()].copy()
print(f"Rows before price filter: {n_before:,} | dropped (item not yet on sale): {n_before - len(sales_long):,} | kept: {len(sales_long):,}")

sales_long = sales_long.sort_values(["item_id", "date"]).reset_index(drop=True)
ITEM_LIST = sorted(sales_long["item_id"].unique())
item_to_code = {k: i for i, k in enumerate(ITEM_LIST)}
sales_long["item_code"] = sales_long["item_id"].map(item_to_code).astype("int32")

# each item's series must be contiguous daily (LSTM windows rely on it)
gaps = sales_long.groupby("item_id")["date"].apply(lambda s: int((s.diff().dropna().dt.days != 1).sum()))
assert gaps.sum() == 0, "Found gaps in daily series"
del selected, prices; gc.collect()

## 5. Đặc trưng lịch sử nhu cầu (chỉ dùng quá khứ)

In [ ]:
g = sales_long.groupby("item_id", sort=False)

# --- historical demand ---
for k in [1, 7, 14, 28]:
    sales_long[f"lag_{k}"] = g["demand"].shift(k)
sales_long["_s1"] = g["demand"].shift(1)
gs = sales_long.groupby("item_id", sort=False)["_s1"]
sales_long["rolling_mean_7"]  = gs.transform(lambda x: x.rolling(7).mean())
sales_long["rolling_mean_28"] = gs.transform(lambda x: x.rolling(28).mean())
sales_long["rolling_std_28"]  = gs.transform(lambda x: x.rolling(28).std())
sales_long.drop(columns="_s1", inplace=True)
sales_long["log_demand"] = np.log1p(sales_long["demand"])

# --- price features (price is known in advance, so it is a legitimate future covariate) ---
gp = sales_long.groupby("item_id", sort=False)["sell_price"]
sales_long["price_rel_28"]   = (sales_long["sell_price"] / gp.transform(lambda x: x.rolling(28, min_periods=1).mean())).clip(0.5, 1.5)
sales_long["price_change_7"] = (sales_long["sell_price"] / gp.shift(7) - 1).fillna(0).clip(-0.5, 0.5)

# --- fixed categorical dtypes (same categories in train/val/test) ---
sales_long["item_cat"]      = pd.Categorical(sales_long["item_id"],   categories=ITEM_LIST)

print("Missing values in key columns (expected: only lag/rolling at the start of each series):")
print(sales_long[["demand", "sell_price", "lag_1", "lag_28", "rolling_mean_28", "price_rel_28"]].isna().sum())

## 6. Chia theo thời gian 70 / 15 / 15

In [ ]:
_tmp = sales_long.dropna(subset=["lag_28", "rolling_mean_28"])
dates = np.sort(_tmp["date"].unique()); n_dates = len(dates)
train_end = pd.Timestamp(dates[int(n_dates * TRAIN_FRAC) - 1])
val_end   = pd.Timestamp(dates[int(n_dates * (TRAIN_FRAC + VAL_FRAC)) - 1])
del _tmp

sales_long["split"] = np.where(sales_long["date"] <= train_end, "train",
                        np.where(sales_long["date"] <= val_end, "val", "test"))
model_df = sales_long.dropna(subset=["lag_28", "rolling_mean_28"]).copy()

train = model_df[model_df["split"] == "train"].copy()
val   = model_df[model_df["split"] == "val"].copy()
test  = model_df[model_df["split"] == "test"].copy()

split_table = pd.DataFrame({
    s: {"from": d["date"].min().date(), "to": d["date"].max().date(), "days": d["date"].nunique(), "rows": len(d)}
    for s, d in [("train", train), ("val", val), ("test", test)]}).T
print(split_table)

## 7. Khám phá dữ liệu

In [ ]:
print("=== Dataset summary (for Report §6) ===")
print("Items:", model_df["item_id"].nunique(), "| Rows (model_df):", f"{len(model_df):,}",
      "| Period:", model_df["date"].min().date(), "->", model_df["date"].max().date())
print("Target 'demand' (units/day):"); print(model_df["demand"].describe().round(3).to_string())
print(f"Share of zero-demand days: {100 * (model_df['demand'] == 0).mean():.1f}%")
print("\nDays per date type (calendar, all days):"); print(cal["date_type"].value_counts().to_string())
print("\nMean demand by date type:"); print(model_df.groupby("date_type")["demand"].mean().round(3).to_string())
print("\nMean demand on SNAP vs non-SNAP days:"); print(model_df.groupby("snap")["demand"].mean().round(3).to_string())

fig, ax = plt.subplots(2, 2, figsize=(15, 9))
daily = model_df.groupby("date")["demand"].sum()
ax[0, 0].plot(daily.index, daily.values, lw=0.8); ax[0, 0].set_title("Total daily demand (sampled items)")
for d in [train_end, val_end]: ax[0, 0].axvline(d, color="red", ls="--", lw=1)
model_df.groupby("day_of_week")["demand"].mean().plot(kind="bar", ax=ax[0, 1], rot=0)
ax[0, 1].set_title("Mean demand by day of week (0 = Monday)")
sns.boxplot(data=model_df, x="date_type", y="demand", order=DATE_TYPES, ax=ax[1, 0], showfliers=False)
ax[1, 0].set_title("Demand by date type (outliers hidden)")
ax[1, 1].hist(model_df["demand"].clip(upper=model_df["demand"].quantile(0.99)), bins=40)
ax[1, 1].set_title("Demand distribution (clipped at 99th pct)")
save_fig("fig_eda.png")

## 8. Bộ biến cho từng mô hình
Lịch sử nhu cầu ở dạng tự nhiên của từng mô hình (LightGBM: lag và thống kê trượt; ARIMA: chính chuỗi; LSTM và N-BEATS: cửa sổ 28 ngày). Biến lịch theo bộ S0–S3, D1, D2.

In [ ]:
HIST_LGB = ["lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28", "rolling_std_28"]
SETS = {m: SETS_NESTED + (["D1", "D2"] if DROP_ONE[m] else []) for m in RUN}
LGB_SETS = {s: HIST_LGB + set_columns(s, "tree") + ["item_cat"] for s in SET_GROUPS}
ARIMA_SETS = {s: set_columns(s, "linear") for s in SET_GROUPS}
LSTM_SETS = {s: dict(hist=["log_demand"] + set_columns(s, "neural"), fut=set_columns(s, "neural")) for s in SET_GROUPS}
NB_SETS = {s: set_columns(s, "neural") for s in SET_GROUPS}
for s in SET_GROUPS:
    print(s, SET_GROUPS[s], "| LightGBM", len(LGB_SETS[s]), "| ARIMA exog", len(ARIMA_SETS[s]),
          "| LSTM hist/fut", len(LSTM_SETS[s]["hist"]), len(LSTM_SETS[s]["fut"]))
print("Chạy:", SETS)

## 9. Khung đánh giá và mô hình cơ sở

In [ ]:
def wape(y, p):  return float(np.abs(y - p).sum() / (np.abs(y).sum() + 1e-8))
def bias(y, p):  return float(np.mean(p - y))

def metrics_row(df, col):
    y, p = df["demand"].to_numpy(float), df[col].to_numpy(float)
    return {"MAE": np.mean(np.abs(y - p)),
            "RMSE": np.sqrt(np.mean((y - p) ** 2)),
            "WAPE_%": 100 * wape(y, p),
            "RMSSE": np.sqrt(np.mean((y - p) ** 2 / df["scale"].to_numpy(float))),
            "Bias": bias(y, p),
            "sMAPE_%(aux)": 100 * np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p) + 1e-8))}

def metrics_table(df, cols):
    return pd.DataFrame({c: metrics_row(df, c) for c in cols}).T.round(4)

def dm_test(df, col1, col2, loss="abs", lag=7):
    """Diebold-Mariano on daily-aggregated loss. stat > 0 => col1 has HIGHER loss (worse) than col2."""
    e1, e2 = df["demand"] - df[col1], df["demand"] - df[col2]
    l1, l2 = (e1.abs(), e2.abs()) if loss == "abs" else (e1 ** 2, e2 ** 2)
    d = (l1 - l2).groupby(df["date"]).sum().to_numpy()
    n, dm = len(d), d.mean()
    var = np.var(d)
    for k in range(1, lag + 1):
        var += 2 * (1 - k / (lag + 1)) * np.mean((d[k:] - dm) * (d[:-k] - dm))
    stat = dm / np.sqrt(var / n)
    return float(stat), float(2 * (1 - stats.norm.cdf(abs(stat))))

# evaluation frame: one row per (item_id, date) of the test period
test_eval = test[["item_id", "date", "demand", "date_type", "snap", "vol_bin"]].copy().reset_index(drop=True)
train_scale = train.groupby("item_id")["demand"].apply(lambda s: float(np.mean(np.diff(s.to_numpy()) ** 2)))
fallback = train_scale[train_scale > 0].median()
test_eval["scale"] = test_eval["item_id"].map(train_scale).replace(0, fallback).fillna(fallback)

# val frame (only used to select LightGBM hyper-parameters)
val_eval = val[["item_id", "date", "demand"]].copy().reset_index(drop=True)
print("Test rows:", len(test_eval), "| Test days:", test_eval["date"].nunique())

In [ ]:
test_eval["Naive"]          = test["lag_1"].to_numpy()
test_eval["Seasonal Naive"] = test["lag_7"].to_numpy()
test_eval["MA28"]           = test["rolling_mean_28"].to_numpy()
print(metrics_table(test_eval, ["Naive", "Seasonal Naive", "MA28"]))

In [ ]:
FAILED = {}

def put(col, preds):
    """Ghép dự báo (item_id, date, pred) vào test_eval; dòng thiếu (≤ 5%) điền bằng MA28 và được ghi lại."""
    global test_eval
    test_eval = attach(test_eval, preds, col, fallback_col="MA28")

def run_col(col, fn):
    """Chạy (hoặc đọc lại) một cột dự báo. Lỗi ở một cột được in ra và ghi vào FAILED, notebook chạy tiếp."""
    try:
        preds = cached(col, fn)
        put(col, preds)
        return preds
    except Exception as e:
        import traceback
        traceback.print_exc()
        FAILED[col] = f"{type(e).__name__}: {e}"
        print(f"  !!! {col} lỗi, bỏ qua: {FAILED[col][:300]}")
        return None

def have(cols):
    return [c for c in cols if c in test_eval.columns]

TEST_DAYS = test_eval["date"].nunique()
VAL_DAYS = val["date"].nunique()
print("Ngày kiểm tra:", TEST_DAYS, "| ngày kiểm định:", VAL_DAYS)

## 10. ARIMA có biến ngoại sinh (họ thống kê)
Bậc (p, 0, q) chọn bằng AIC trên tập huấn luyện với S0 và giữ nguyên cho mọi bộ; biến lịch vào mô hình dưới dạng biến giả. Dự báo một bước với tham số cố định (bộ lọc Kalman), không huấn luyện lại. Chạy trên CPU, song song theo mặt hàng.

In [ ]:
try:
    if RUN["ARIMA"]:
        arima_input = sales_long[["item_id", "date", "demand"] + sorted(set(sum(ARIMA_SETS.values(), [])))]
        ARIMA_ORDERS = None
        order_path = os.path.join(PRED_DIR, "arima_orders.json")
        if os.path.exists(order_path):
            import json
            ARIMA_ORDERS = {k: tuple(v) for k, v in json.load(open(order_path)).items()}
        for s in SETS["ARIMA"]:
            def fit_one(s=s):
                out = run_arima_panel(arima_input, ARIMA_SETS[s], train_end, orders=ARIMA_ORDERS,
                                      candidates=ARIMA_CANDIDATES, n_jobs=ARIMA_N_JOBS, fallback_end=val_end)
                return out
            preds = run_col(f"ARIMA-{s}", fit_one)
            if preds is None:
                continue
            if s == "S0" and ARIMA_ORDERS is None:
                import json
                ARIMA_ORDERS = preds.attrs.get("orders") or None
                if ARIMA_ORDERS:
                    json.dump({k: list(v) for k, v in ARIMA_ORDERS.items()}, open(order_path, "w"))
        print(metrics_table(test_eval, have([f"ARIMA-{s}" for s in SETS["ARIMA"]])))
except Exception as _e:
    import traceback
    traceback.print_exc()
    FAILED['ARIMA'] = f"{type(_e).__name__}: {_e}"
    print("!!! ARIMA lỗi, bỏ qua mô hình này; các ô sau vẫn chạy.")


## 11. LightGBM (họ cây)
Tweedie, chọn từ 3 cấu hình (15, 31, 63 lá) theo WAPE kiểm định, dừng sớm 100 vòng — giống v2, chỉ thay bộ biến. Ô cuối minh họa vì sao phải phân rã ngày tháng: dùng thẳng chỉ số ngày và năm (bộ R) so với S1.

In [ ]:
try:
    import lightgbm as lgb

    LGB_GRID = [dict(num_leaves=15, min_child_samples=50),
                dict(num_leaves=31, min_child_samples=20),
                dict(num_leaves=63, min_child_samples=20)]
    lgb_models, lgb_val_log = {}, []

    def fit_lgb(cols, tag):
        best = None
        for params in LGB_GRID:
            m = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=1.2, n_estimators=3000,
                                  learning_rate=0.03, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1, **params)
            m.fit(train[cols], train["demand"], eval_set=[(val[cols], val["demand"])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
            v = wape(val["demand"].to_numpy(), np.clip(m.predict(val[cols]), 0, None))
            lgb_val_log.append({"set": tag, **params, "best_iter": m.best_iteration_, "val_WAPE_%": round(100 * v, 3)})
            if best is None or v < best[0]:
                best = (v, m)
        lgb_models[tag] = best[1]
        return pd.DataFrame({"item_id": test["item_id"].to_numpy(), "date": test["date"].to_numpy(),
                             "pred": np.clip(best[1].predict(test[cols]), 0, None)})

    if RUN["LightGBM"]:
        for s in SETS["LightGBM"]:
            run_col(f"LightGBM-{s}", lambda s=s: fit_lgb(LGB_SETS[s], s))
        print(pd.DataFrame(lgb_val_log).to_string(index=False))
        print(metrics_table(test_eval, have([f"LightGBM-{s}" for s in SETS["LightGBM"]])))

    if RUN["LightGBM"] and RUN_RAW_DATE_DEMO:
        for df_ in (train, val, test):
            df_["date_index"] = (df_["date"] - pd.Timestamp("2011-01-29")).dt.days
            df_["year"] = df_["date"].dt.year
        run_col("LightGBM-R", lambda: fit_lgb(HIST_LGB + ["date_index", "year", "item_cat"], "R"))
        print("Ngày tháng dùng thẳng (R) so với phân rã (S1):")
        print(metrics_table(test_eval, have(["LightGBM-S0", "LightGBM-R", "LightGBM-S1"])))
except Exception as _e:
    import traceback
    traceback.print_exc()
    FAILED['LightGBM'] = f"{type(_e).__name__}: {_e}"
    print("!!! LightGBM lỗi, bỏ qua mô hình này; các ô sau vẫn chạy.")


## 12. N-BEATS có biến ngoại sinh (họ kiến trúc mới)
Chạy trước LSTM: PyTorch (N-BEATS) trả lại bộ nhớ GPU sau khi xong, còn TensorFlow (LSTM) giữ bộ nhớ đến hết phiên.
NBEATSx của thư viện neuralforecast, h = 1, cửa sổ 28 ngày, biến lịch của ngày dự báo đưa vào `futr_exog_list`, mất mát MSE (dự báo kỳ vọng). Độ rộng MLP chọn từ 3 cấu hình trên tập kiểm định (bộ S3, seed đầu); dự báo tập kiểm tra bằng `cross_validation` bước 1 ngày, không huấn luyện lại; trung bình 3 seed.

In [ ]:
try:
    if RUN["N-BEATS"]:
        try:
            import neuralforecast  # noqa: F401
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuralforecast"], check=True)
        import logging
        logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
        logging.getLogger("lightning").setLevel(logging.ERROR)

        # Thiết bị: GPU nếu còn đủ bộ nhớ, ngược lại CPU. Lỗi CUBLAS_STATUS_NOT_INITIALIZED / out of memory
        # thường do TensorFlow (LSTM) đã giữ bộ nhớ GPU trong cùng phiên.
        NB_DEVICE = {"accelerator": "auto"}
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                _free, _total = torch.cuda.mem_get_info()
                print(f"GPU còn trống {_free / 2**30:.1f} / {_total / 2**30:.1f} GB")
                if _free < 2 * 2**30:
                    NB_DEVICE["accelerator"] = "cpu"
                    print("GPU gần hết bộ nhớ → N-BEATS chạy trên CPU (chậm hơn). Muốn dùng GPU: Runtime → Restart session, chạy lại; mô hình đã xong được đọc lại từ PRED_DIR.")
            else:
                NB_DEVICE["accelerator"] = "cpu"
        except Exception as _e:
            NB_DEVICE["accelerator"] = "cpu"
            print("Không khởi tạo được GPU cho PyTorch:", str(_e)[:150], "→ dùng CPU")

        nb_all_cols = sorted(set(sum(NB_SETS.values(), [])))
        nb_df_full = build_nf_frame(sales_long, nb_all_cols)

        def _is_gpu_error(e):
            s = str(e).lower()
            return any(k in s for k in ("cuda", "cublas", "cudnn", "out of memory"))

        def nb_cv(frame, cols, test_size, val_size, seed, params):
            """Gọi NBEATSx. Lỗi GPU → chuyển hẳn sang CPU và chạy lại; lỗi khác (thường do chuỗi ngắn) → thử val_size = 28."""
            try:
                return nbeatsx_cross_validation(frame, cols, test_size, val_size, seed, dict(params, **NB_DEVICE))
            except Exception as e:
                if _is_gpu_error(e) and NB_DEVICE["accelerator"] != "cpu":
                    print("  Lỗi GPU khi chạy NBEATSx:", str(e)[:120], "→ chuyển sang CPU")
                    NB_DEVICE["accelerator"] = "cpu"
                    return nb_cv(frame, cols, test_size, val_size, seed, params)
                if _is_gpu_error(e):
                    raise
                print("  Cảnh báo NBEATSx:", str(e)[:200], "→ thử lại với val_size = 28")
                return nbeatsx_cross_validation(frame, cols, test_size, 28, seed, dict(params, **NB_DEVICE))

        width_path = os.path.join(PRED_DIR, "nbeats_width.json")
        if os.path.exists(width_path):
            import json
            NB_WIDTH = json.load(open(width_path))
        else:
            nb_df_val = nb_df_full[nb_df_full["ds"] <= val_end]
            long_enough = nb_df_val.groupby("unique_id")["ds"].transform("size") >= 2 * VAL_DAYS + LOOKBACK
            nb_df_val = nb_df_val[long_enough]                      # chọn cấu hình trên chuỗi đủ dài
            tune = []
            for w in NB_GRID:
                t0 = time.time()
                p = nb_cv(nb_df_val[["unique_id", "ds", "y"] + NB_SETS["S3"]], NB_SETS["S3"],
                          VAL_DAYS, VAL_DAYS, NB_SEEDS[0], dict(NB_PARAMS, mlp_width=w))
                v = val_eval.merge(p.rename(columns={"unique_id": "item_id", "ds": "date"}), on=["item_id", "date"], how="inner")
                tune.append({"mlp_width": w, "val_WAPE_%": round(100 * wape(v["demand"], v["pred"].clip(lower=0)), 3),
                             "rows": len(v), "sec": round(time.time() - t0)})
                print(tune[-1])
            NB_WIDTH = min(tune, key=lambda r: r["val_WAPE_%"])["mlp_width"]
            import json
            json.dump(NB_WIDTH, open(width_path, "w"))
        print("Độ rộng MLP N-BEATS:", NB_WIDTH)

        for s in SETS["N-BEATS"]:
            def fit_one(s=s):
                cols = NB_SETS[s]
                runs = []
                for seed in NB_SEEDS:
                    p = nb_cv(nb_df_full[["unique_id", "ds", "y"] + cols], cols, TEST_DAYS, VAL_DAYS, seed,
                              dict(NB_PARAMS, mlp_width=NB_WIDTH))
                    runs.append(p.set_index(["unique_id", "ds"])["pred"])
                out = pd.concat(runs, axis=1).mean(axis=1).rename("pred").reset_index()
                return out.rename(columns={"unique_id": "item_id", "ds": "date"})
            run_col(f"N-BEATS-{s}", fit_one)
        print(metrics_table(test_eval, have([f"N-BEATS-{s}" for s in SETS["N-BEATS"]])))
        try:                                                         # trả bộ nhớ GPU cho LSTM (TensorFlow) chạy sau
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
except Exception as _e:
    import traceback
    traceback.print_exc()
    FAILED['N-BEATS'] = f"{type(_e).__name__}: {_e}"
    print("!!! N-BEATS lỗi, bỏ qua mô hình này; các ô sau vẫn chạy.")


## 13. LSTM (họ mạng hồi quy)
Kiến trúc hai nhánh như v2: nhánh lịch sử đọc cửa sổ 28 ngày (log doanh số và biến lịch của các ngày đó), nhánh biết trước nhận biến lịch của ngày dự báo, embedding mặt hàng, đầu ra softplus, mất mát Poisson. Số đơn vị LSTM chọn từ 3 cấu hình trên tập kiểm định (bộ S3, seed đầu), rồi huấn luyện mọi bộ với 3 seed.

In [ ]:
# ---- standardise continuous inputs with TRAIN statistics only ----
SCALE_COLS = ["log_demand"]
lstm_df = sales_long.copy()
tr_rows = lstm_df["split"] == "train"
mu, sd = lstm_df.loc[tr_rows, SCALE_COLS].mean(), lstm_df.loc[tr_rows, SCALE_COLS].std().replace(0, 1)
lstm_df[SCALE_COLS] = (lstm_df[SCALE_COLS] - mu) / sd

def build_sequences(df, hist_cols, fut_cols, lookback, horizon):
    """Vectorised, per-item sliding windows.
    Xh: (N, lookback, n_hist)  past window   | Xf: (N, horizon, n_fut) calendar etc. of the forecast days
    Y : (N, horizon) raw demand of the forecast days | start/end: first/last forecast date of each window"""
    out = {k: [] for k in ["Xh", "Xf", "item", "Y", "start", "end", "item_id"]}
    for item_id, g in df.groupby("item_id", sort=False):
        g = g.sort_values("date")
        n = len(g); m = n - lookback - horizon + 1
        if m <= 0:
            continue
        H_arr = g[hist_cols].to_numpy(np.float32)
        out["Xh"].append(sliding_window_view(H_arr, lookback, axis=0)[:m].transpose(0, 2, 1))
        y = g["demand"].to_numpy(np.float32)
        out["Y"].append(sliding_window_view(y, horizon)[lookback:lookback + m])
        if fut_cols:
            F_arr = g[fut_cols].to_numpy(np.float32)
            out["Xf"].append(sliding_window_view(F_arr, horizon, axis=0)[lookback:lookback + m].transpose(0, 2, 1))
        dts = g["date"].to_numpy()
        out["start"].append(dts[lookback:lookback + m])
        out["end"].append(dts[lookback + horizon - 1: lookback + horizon - 1 + m])
        out["item"].append(np.full(m, g["item_code"].iloc[0], dtype=np.int32))
        out["item_id"].append(np.full(m, item_id, dtype=object))
    seq = {k: (np.concatenate(v) if len(v) else None) for k, v in out.items()}
    seq["item"] = seq["item"].reshape(-1, 1)
    return seq

def split_masks(seq, t_end, v_end):
    t_end, v_end = np.datetime64(t_end), np.datetime64(v_end)
    tr = seq["end"] <= t_end                                     # whole target window inside train
    va = (seq["start"] > t_end) & (seq["end"] <= v_end)          # inside validation
    te = seq["start"] > v_end                                    # forecast origin in test
    return tr, va, te

def make_inputs(seq, mask):
    d = {"hist": seq["Xh"][mask], "item": seq["item"][mask]}
    if seq["Xf"] is not None:
        d["future"] = seq["Xf"][mask]
    return d

def attach_preds(base, seq, mask, pred_1d, col):
    tmp = pd.DataFrame({"item_id": seq["item_id"][mask],
                        "date": pd.to_datetime(seq["start"][mask]).astype("datetime64[ns]"),
                        col: np.asarray(pred_1d, dtype=float)})
    out = base.merge(tmp, on=["item_id", "date"], how="left")
    assert out[col].notna().all(), f"{col}: some test rows have no LSTM prediction"
    assert len(out) == len(base)
    return out

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    for _g in tf.config.list_physical_devices("GPU"):          # để dành bộ nhớ GPU cho N-BEATS (PyTorch)
        try:
            tf.config.experimental.set_memory_growth(_g, True)
        except RuntimeError:
            pass
    print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

    def build_lstm(n_hist, n_fut, n_items, units, horizon=1, lookback=LOOKBACK):
        hist_in = layers.Input((lookback, n_hist), name="hist")
        item_in = layers.Input((1,), dtype="int32", name="item")
        emb = layers.Flatten()(layers.Embedding(n_items, 4)(item_in))
        h = layers.LSTM(units[0], return_sequences=True)(hist_in)
        h = layers.Dropout(DROPOUT)(h)
        h = layers.LSTM(units[1])(h)
        h = layers.Dropout(DROPOUT)(h)
        parts, inputs = [h, emb], [hist_in, item_in]
        if n_fut > 0:
            fut_in = layers.Input((horizon, n_fut), name="future")
            parts.append(layers.Flatten()(fut_in)); inputs.append(fut_in)
        z = layers.Dense(32, activation="relu")(layers.Concatenate()(parts))
        out = layers.Dense(horizon, activation="softplus")(z)
        model = Model(inputs, out)
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=LSTM_LOSS)
        return model

    def make_callbacks():
        return [EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)]

    def train_lstm(seq, tr, va, spec, units, seed):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
        model = build_lstm(len(spec["hist"]), len(spec["fut"]), len(ITEM_LIST), units)
        model.fit(make_inputs(seq, tr), seq["Y"][tr], validation_data=(make_inputs(seq, va), seq["Y"][va]),
                  epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=make_callbacks(), verbose=0)
        return model

    LSTM_UNITS = None
    if RUN["LSTM"]:
        unit_path = os.path.join(PRED_DIR, "lstm_units.json")
        if os.path.exists(unit_path):
            import json
            LSTM_UNITS = tuple(json.load(open(unit_path)))
        else:
            spec = LSTM_SETS["S3"]
            seq = build_sequences(lstm_df, spec["hist"], spec["fut"], LOOKBACK, HORIZON)
            tr, va, te = split_masks(seq, train_end, val_end)
            tune = []
            for units in LSTM_GRID:
                t0 = time.time()
                mdl = train_lstm(seq, tr, va, spec, units, SEEDS[0])
                pv = np.clip(mdl.predict(make_inputs(seq, va), batch_size=4096, verbose=0)[:, 0], 0, None)
                tune.append({"units": units, "val_WAPE_%": round(100 * wape(seq["Y"][va][:, 0], pv), 3), "sec": round(time.time() - t0)})
                print(tune[-1])
            LSTM_UNITS = min(tune, key=lambda r: r["val_WAPE_%"])["units"]
            import json
            json.dump(list(LSTM_UNITS), open(unit_path, "w"))
            del seq; gc.collect()
        print("Cấu hình LSTM:", LSTM_UNITS)

        for s in SETS["LSTM"]:
            def fit_one(s=s):
                spec = LSTM_SETS[s]
                seq = build_sequences(lstm_df, spec["hist"], spec["fut"], LOOKBACK, HORIZON)
                tr, va, te = split_masks(seq, train_end, val_end)
                preds = []
                for seed in SEEDS:
                    mdl = train_lstm(seq, tr, va, spec, LSTM_UNITS, seed)
                    preds.append(np.clip(mdl.predict(make_inputs(seq, te), batch_size=4096, verbose=0)[:, 0], 0, None))
                return pd.DataFrame({"item_id": seq["item_id"][te],
                                     "date": pd.to_datetime(seq["start"][te]).astype("datetime64[ns]"),
                                     "pred": np.mean(preds, axis=0)})
            run_col(f"LSTM-{s}", fit_one)
        print(metrics_table(test_eval, have([f"LSTM-{s}" for s in SETS["LSTM"]])))
except Exception as _e:
    import traceback
    traceback.print_exc()
    FAILED['LSTM'] = f"{type(_e).__name__}: {_e}"
    print("!!! LSTM lỗi, bỏ qua mô hình này; các ô sau vẫn chạy.")


### Ghép dự báo đã lưu từ các phiên trước
Mỗi mô hình có thể chạy ở một phiên Colab riêng (tắt các mô hình khác trong `RUN`); ô này ghép mọi dự báo đã lưu trong `PRED_DIR` chưa có trong phiên hiện tại.

In [ ]:
for _m in ["ARIMA", "LightGBM", "LSTM", "N-BEATS"]:
    for _s in SETS_NESTED + ["D1", "D2"] + (["R"] if _m == "LightGBM" else []):
        _col = f"{_m}-{_s}"
        _path = os.path.join(PRED_DIR, f"{_col}.parquet")
        if _col not in test_eval.columns and os.path.exists(_path):
            try:
                put(_col, pd.read_parquet(_path))
                print("  ghép từ dự báo đã lưu:", _col)
            except Exception as _e:
                FAILED[_col] = f"{type(_e).__name__}: {_e}"
print("Các cột dự báo hiện có:", [c for c in test_eval.columns if any(c.startswith(m + "-") for m in ["ARIMA", "LightGBM", "LSTM", "N-BEATS"])])

## 14. Kết quả
### 14.1 Bảng tổng hợp — mọi mô hình, mọi bộ, cùng 28.700 dòng kiểm tra

In [ ]:
MODELS = [m for m in ["ARIMA", "LightGBM", "LSTM", "N-BEATS"] if any(c.startswith(m + "-S") for c in test_eval.columns)]
SETS = {m: [s for s in SETS_NESTED + ["D1", "D2"] if f"{m}-{s}" in test_eval.columns] for m in ["ARIMA", "LightGBM", "LSTM", "N-BEATS"]}
if FAILED:
    print("Các cột bị lỗi (xem traceback ở ô tương ứng):", FAILED)
print("Mô hình có kết quả:", MODELS, "| dòng điền bằng MA28:", {k: v for k, v in test_eval.attrs.get("filled", {}).items() if v})
cols = ["Naive", "Seasonal Naive", "MA28"] + [f"{m}-{s}" for m in MODELS for s in SETS[m] if f"{m}-{s}" in test_eval]
cols += [c for c in ["LightGBM-R"] if c in test_eval]
results = metrics_table(test_eval, cols)
print(results.to_string())
wape_grid = pd.DataFrame({m: {s: results.loc[f"{m}-{s}", "WAPE_%"] for s in SETS[m] if f"{m}-{s}" in results.index}
                          for m in MODELS}).T
print("\nWAPE (%) — mô hình × bộ biến"); print(wape_grid)

### 14.2 Tác động của từng nhóm yếu tố lịch (RQ1, RQ2)
`delta` < 0 nghĩa là thêm nhóm làm giảm sai số. "Tăng thêm" = thêm theo thứ tự G1 → G2 → G3; "biên" = bỏ đúng nhóm đó khỏi bộ đầy đủ. "Ngày liên quan": G2 trên ngày lễ, sự kiện và trước sự kiện; G3 trên ngày SNAP. DM > 0 với p < 0,05 nghĩa là nhóm giúp có ý nghĩa.

In [ ]:
effects = group_effects(test_eval, MODELS)
print(effects.to_string(index=False))
mat = effect_matrix(effects, kind="biên", scope="mọi ngày", value="delta (%)")
mat_inc = effect_matrix(effects, kind="tăng thêm", scope="mọi ngày", value="delta (%)")
print("\nTác động biên (% thay đổi WAPE) — mô hình × nhóm"); print(mat)
print("\nTác động tăng thêm (% thay đổi WAPE)"); print(mat_inc)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, (m_, t_) in zip(ax, [(mat_inc, "Tăng thêm (G1 → G2 → G3)"), (mat, "Biên (bỏ một nhóm khỏi S3)")]):
    sns.heatmap(m_.reindex(columns=["G1", "G2", "G3"]), annot=True, fmt=".2f", cmap="RdBu", center=0, ax=a,
                cbar_kws={"label": "% thay đổi WAPE"})
    a.set_title(t_); a.set_xlabel("nhóm yếu tố lịch"); a.set_ylabel("")
save_fig("fig_group_effects_heatmap.png")

### 14.3 Khoảng tin cậy bootstrap cho tác động biên trên ngày liên quan (RQ1, RQ3)
Lấy mẫu lại theo khối tuần; `diff b-a` < 0 nghĩa là bộ đầy đủ S3 tốt hơn bộ thiếu nhóm.

In [ ]:
boot = []
for m in MODELS:
    for g, before in [("G1", "D1"), ("G2", "D2"), ("G3", "S2")]:
        a, b = f"{m}-{before}", f"{m}-S3"
        if a not in test_eval or b not in test_eval:
            continue
        sub = test_eval[relevant_mask(test_eval, g)]
        r = block_bootstrap_wape_diff(sub, a, b, None, N_BOOT, SEED).assign(model=m, group=g)
        boot.append(r)
boot = pd.concat(boot, ignore_index=True) if boot else pd.DataFrame()
print(boot[["model", "group", "WAPE a (%)", "WAPE b (%)", "diff b-a (pp)", "CI low", "CI high", "CI excludes 0", "days"]].to_string(index=False))

### 14.4 Sai số theo loại ngày: S0 so với S3 của từng mô hình (RQ3)

In [ ]:
def grouped_metric(df, by, cols, fn):
    return pd.DataFrame({c: df.groupby(by).apply(lambda x: fn(x["demand"].to_numpy(float), x[c].to_numpy(float))) for c in cols})

dt_cols = ["Seasonal Naive"] + [f"{m}-{s}" for m in MODELS for s in ["S0", "S3"] if f"{m}-{s}" in test_eval]
wape_dt = (100 * grouped_metric(test_eval, "date_type", dt_cols, wape)).reindex(DATE_TYPES).round(2)
bias_dt = grouped_metric(test_eval, "date_type", dt_cols, bias).reindex(DATE_TYPES).round(3)
n_dt = test_eval.groupby("date_type")["date"].nunique().reindex(DATE_TYPES).rename("test_days")
print("WAPE (%) theo loại ngày"); print(pd.concat([n_dt, wape_dt], axis=1).to_string())
print("\nBias theo loại ngày"); print(bias_dt.to_string())
wape_snap = (100 * grouped_metric(test_eval, "snap", dt_cols, wape)).round(2); wape_snap.index = ["non-SNAP", "SNAP"][:len(wape_snap)]
print("\nWAPE (%) theo ngày SNAP"); print(wape_snap.to_string())
wape_dt.plot(kind="bar", figsize=(13, 5), rot=0, title="WAPE (%) theo loại ngày, S0 và S3 của từng mô hình")
save_fig("fig_wape_by_date_type_v4.png")

### 14.5 So sánh giữa các họ mô hình ở bộ đầy đủ S3 và chi phí tính toán (RQ2, RQ4)

In [ ]:
pairs = [(f"{a}-S3", f"{b}-S3") for i, a in enumerate(MODELS) for b in MODELS[i + 1:]
         if f"{a}-S3" in test_eval and f"{b}-S3" in test_eval]
dm_models = dm_pairs_table(test_eval, pairs) if pairs else pd.DataFrame()
print(dm_models.to_string(index=False))
times = pd.DataFrame(TIMES)
if len(times):
    times["model"] = times["column"].str.rsplit("-", n=1).str[0]
    print("\nThời gian chạy (giây, gồm mọi seed và cấu hình; chỉ các cột chạy trong phiên này):")
    print(times.to_string(index=False))

## 15. Lưu kết quả

In [ ]:
results.to_csv(os.path.join(OUT_DIR, "v4_results_main.csv"))
wape_grid.to_csv(os.path.join(OUT_DIR, "v4_wape_model_by_set.csv"))
effects.to_csv(os.path.join(OUT_DIR, "v4_group_effects.csv"), index=False)
if len(boot): boot.to_csv(os.path.join(OUT_DIR, "v4_bootstrap_marginal.csv"), index=False)
wape_dt.to_csv(os.path.join(OUT_DIR, "v4_wape_by_date_type.csv"))
dm_models.to_csv(os.path.join(OUT_DIR, "v4_dm_between_models.csv"), index=False)
if len(times): times.to_csv(os.path.join(OUT_DIR, "v4_times.csv"), index=False)
test_eval.to_csv(os.path.join(OUT_DIR, "v4_test_predictions.csv"), index=False)
if USE_DRIVE:
    import shutil
    shutil.copytree(OUT_DIR, "/content/drive/MyDrive/M5_project/outputs_v4", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.zip", "*.csv.gz"))
print("Đã lưu vào", OUT_DIR)

# Gom các bảng kết quả (không gồm tệp dự báo lớn) thành v4_results.zip để tải về
import glob, shutil
pd.DataFrame([{"column": k, "error": v} for k, v in FAILED.items()]).to_csv(os.path.join(OUT_DIR, "v4_failed.csv"), index=False)
pd.Series(test_eval.attrs.get("filled", {}), name="rows_filled_with_MA28").to_csv(os.path.join(OUT_DIR, "v4_filled_rows.csv"))
_zip_dir = os.path.join(OUT_DIR, "v4_results")
os.makedirs(_zip_dir, exist_ok=True)
for f in glob.glob(os.path.join(OUT_DIR, "v4_*.csv")) + glob.glob(os.path.join(OUT_DIR, "fig_*.png")):
    if "test_predictions" not in f:
        shutil.copy(f, _zip_dir)
_zip = shutil.make_archive(_zip_dir, "zip", _zip_dir)
print("Tệp kết quả để gửi:", _zip)
try:
    from google.colab import files
    files.download(_zip)
except Exception:
    pass

## 16. Đưa kết quả vào báo cáo
* **RQ1 – mỗi nhóm yếu tố lịch tác động bao nhiêu?** Bảng `effects` (cột `delta (%)`, `DM abs`, `p abs`), loại "biên" là con số chính, "tăng thêm" là bổ sung; khoảng tin cậy ở `boot`.
* **RQ2 – tác động khác nhau thế nào giữa các họ mô hình?** Hình nhiệt `fig_group_effects_heatmap.png` (mô hình × nhóm).
* **RQ3 – tác động rơi vào ngày nào?** Bảng theo loại ngày và theo SNAP (14.4); tác động trên "ngày liên quan" trong `effects`.
* **Minh họa phân rã ngày tháng:** so sánh LightGBM-R với LightGBM-S1 (mục 11).
* **RQ4 – chi phí:** bảng thời gian (14.5), ghi rõ phần cứng.